# House Price Prediction - Model Training and Evaluation

## Objective

The objective of this notebook is to train and evaluate multiple regression models for predicting house prices.

Rather than selecting a single algorithm immediately, several models will be compared using the same preprocessing pipeline. This approach allows us to identify the model that provides the best predictive performance on unseen data.

The following models will be evaluated:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

Each model will be assessed using three common regression metrics:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

## 1. Load and Prepare the Dataset

The dataset is loaded and the preprocessing steps from the previous notebook are applied.

This ensures that all models are trained using identical preprocessing procedures.

In [ ]:
house_data = pd.read_csv("../data/raw/dataset_2.csv")

house_data[["Area_SqFt", "Rooms"]] = (
    SimpleImputer(strategy="median")
    .fit_transform(house_data[["Area_SqFt", "Rooms"]])
)

house_data[["Furnishing"]] = (
    SimpleImputer(strategy="most_frequent")
    .fit_transform(house_data[["Furnishing"]])
)

In [ ]:
X = house_data.drop("Price", axis=1)

y = house_data["Price"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
numerical_features = [
    "Area_SqFt",
    "Rooms",
    "Build_Year"
]

categorical_features = [
    "Location",
    "Street_Type",
    "Furnishing",
    "Property_Type",
    "Has_Pool"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

## 2. Define the Models

Four regression algorithms are selected for comparison.

These models range from a simple linear approach to more advanced ensemble methods, allowing us to evaluate how model complexity affects predictive performance.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

## 3. Train and Evaluate the Models

Each model is combined with the preprocessing pipeline and trained using the training dataset.

Predictions are generated on the testing dataset and evaluated using MAE, RMSE, and R² Score.

In [ ]:
results = []

for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    rmse = root_mean_squared_error(y_test, predictions)

    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R² Score": r2
    })

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R² Score",
    ascending=False
)

results_df

## 4. Visual Comparison

A visual comparison makes it easier to identify the best-performing model.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.bar(
    results_df["Model"],
    results_df["R² Score"]
)

plt.title("Model Comparison")

plt.ylabel("R² Score")

plt.xticks(rotation=20)

plt.show()

## 5. Discussion

The comparison demonstrates that different machine learning algorithms perform differently on the same dataset.

Ensemble methods such as Random Forest and Gradient Boosting often outperform simpler models because they can capture complex, non-linear relationships between property characteristics and house prices.

The model with the highest R² Score and the lowest prediction errors will be selected for hyperparameter tuning in the next stage of the project.

# Conclusions

Several regression algorithms were successfully trained and evaluated using a consistent preprocessing pipeline.

The use of Scikit-Learn's Pipeline and ColumnTransformer ensures that preprocessing is applied consistently and prevents data leakage.

Based on the evaluation metrics, the best-performing model will be optimized through hyperparameter tuning before being deployed in the final application.